# EZhire Model B - RoBERTa
Trains sentence-transformers/all-roberta-large-v1 with contextual chunking.

In [ ]:
!pip install -q sentence-transformers datasets scikit-learn nltk PyMuPDF einops accelerate

In [ ]:
import os, re, warnings
import numpy as np
import pandas as pd
import nltk
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, util, InputExample, losses
from sentence_transformers.evaluation import SentenceEvaluator
from torch.utils.data import DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_absolute_error, r2_score
from scipy.stats import spearmanr, pearsonr
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

warnings.filterwarnings("ignore")
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_DIR = os.path.abspath(".")
MODEL_DIR = os.path.join(BASE_DIR, "saved_models")
os.makedirs(MODEL_DIR, exist_ok=True)

STOP_WORDS = set(stopwords.words("english"))
print(f"Device: {DEVICE}")

In [ ]:
ATS_MIN, ATS_MAX = 18.3, 90.7

def split_sep(text):
    if not isinstance(text, str): text = str(text)
    if '[SEP]' in text:
        a, b = text.split('[SEP]', 1)
        return a.strip(), b.strip()
    mid = len(text) // 2
    return text[:mid].strip(), text[mid:].strip()

def raw_text(text):
    if not isinstance(text, str): text = str(text)
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    lines = [re.sub(r'[ \t\f\v]+', ' ', l).strip() for l in text.split('\n')]
    return ' '.join(l for l in lines if l)

def clean_text(text):
    if not isinstance(text, str): text = str(text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text.lower())
    tokens = word_tokenize(re.sub(r'\s+', ' ', text).strip())
    return ' '.join(t for t in tokens if t not in STOP_WORDS and len(t) > 1)

def normalize_score(series):
    v = pd.to_numeric(series, errors='coerce').astype(float)
    return ((v - ATS_MIN) / (ATS_MAX - ATS_MIN)).clip(0, 1)

def denormalize_score(v):
    return np.asarray(v, float) * (ATS_MAX - ATS_MIN) + ATS_MIN

def build_df(src):
    splits = src['text'].apply(split_sep)
    ats = pd.to_numeric(src['ats_score'], errors='coerce').fillna(ATS_MIN)
    return pd.DataFrame({
        'resume_raw': splits.apply(lambda x: raw_text(x[0])),
        'jd_raw': splits.apply(lambda x: raw_text(x[1])),
        'resume_clean': splits.apply(lambda x: clean_text(x[0])),
        'jd_clean': splits.apply(lambda x: clean_text(x[1])),
        'original_label': src['original_label'].values,
        'ats_score_raw': ats.values,
        'ground_truth': normalize_score(ats).values
    }).dropna(subset=['resume_raw', 'jd_raw']).reset_index(drop=True)

ds = load_dataset('0xnbk/resume-ats-score-v1-en')
df_train = ds['train'].to_pandas()
df_val = ds['validation'].to_pandas()

df_tr = build_df(df_train)
df_vl = build_df(df_val)
print(f'Train: {len(df_tr)} | Val: {len(df_vl)}')
print(f'GT range train: {df_tr["ground_truth"].min():.3f} - {df_tr["ground_truth"].max():.3f}')
df_tr.head(2)

## Shared Chunking Infrastructure
Used by Model B (512-token model).

In [ ]:
CHUNK_OVERLAP = 38
MAX_RESUME_CHUNKS = 10
MAX_JD_CHUNKS = 8
MAX_TRAIN_PAIRS = 4
MAX_EVAL_PAIRS = 2
TOP_K_CHUNK_PAIRS = 5
ENCODE_BATCH = 16 if DEVICE == 'cuda' else 8
DOC_EVAL_N = 200
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(RANDOM_SEED)

def extract_first_sentence(text, max_chars=150):
    t = raw_text(text)
    m = re.search(r'(?<=[a-zA-Z0-9])[.!?]', t)
    if m and m.start() > 10:
        sentence = t[:m.start() + 1].strip()
    else:
        sentence = t[:max_chars].strip()
    return sentence[:max_chars]

def token_chunk_body(body, tokenizer, max_tokens=512, overlap=CHUNK_OVERLAP, prefix_tokens=0):
    body = raw_text(body)
    if not body:
        return []
    ids = tokenizer.encode(body, add_special_tokens=False, truncation=False)
    effective_max = max(64, max_tokens - prefix_tokens - 2)
    if len(ids) <= effective_max:
        return [body]
    overlap = min(overlap, effective_max // 2)
    step = effective_max - overlap
    chunks = []
    for start in range(0, len(ids), step):
        piece = ids[start:start + effective_max]
        chunk = raw_text(tokenizer.decode(piece, skip_special_tokens=True))
        if chunk:
            chunks.append(chunk)
        if start + effective_max >= len(ids):
            break
    return chunks

def make_text_chunks(text, tokenizer, model_max_tokens=512, overlap=CHUNK_OVERLAP,
                     max_chunks=MAX_RESUME_CHUNKS, source_label='DOCUMENT'):
    text = raw_text(text)
    if not text:
        return []
    doc_ctx = extract_first_sentence(text)
    prefix = f'[DOC]: {doc_ctx} [{source_label}] ' if doc_ctx else f'[{source_label}] '
    prefix_tokens = len(tokenizer.encode(prefix, add_special_tokens=False))
    body_chunks = token_chunk_body(
        text, tokenizer,
        max_tokens=model_max_tokens,
        overlap=overlap,
        prefix_tokens=prefix_tokens
    )
    chunks = [f'{prefix}{c}' for c in body_chunks]
    if len(chunks) > max_chunks:
        keep = np.linspace(0, len(chunks) - 1, max_chunks, dtype=int).tolist()
        chunks = [chunks[i] for i in keep]
    fallback = text[:2000]
    return chunks or [f'{prefix}{fallback}']

def select_top_chunk_pairs(r_chunks, j_chunks, max_pairs=MAX_TRAIN_PAIRS):
    if len(r_chunks) * len(j_chunks) <= max_pairs:
        return [(r, j) for r in r_chunks for j in j_chunks]
    try:
        vec = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), max_features=5000)
        mat = vec.fit_transform(r_chunks + j_chunks)
        sims = cosine_similarity(mat[:len(r_chunks)], mat[len(r_chunks):])
        ranked = np.argsort(sims.reshape(-1))[::-1]
    except Exception:
        ranked = range(len(r_chunks) * len(j_chunks))
    selected, seen = [], set()
    for idx in ranked:
        i, j = int(idx) // len(j_chunks), int(idx) % len(j_chunks)
        if (i, j) not in seen:
            selected.append((r_chunks[i], j_chunks[j]))
            seen.add((i, j))
        if len(selected) >= max_pairs:
            break
    return selected or [(r_chunks[0], j_chunks[0])]

def build_chunked_examples(frame, tokenizer, max_pairs=MAX_TRAIN_PAIRS, model_max_tokens=512):
    examples = []
    total_rows = len(frame)
    for idx, (_, row) in enumerate(frame.iterrows()):
        rc = make_text_chunks(row['resume_raw'], tokenizer,
                              model_max_tokens=model_max_tokens,
                              max_chunks=MAX_RESUME_CHUNKS,
                              source_label='RESUME')
        jc = make_text_chunks(row['jd_raw'], tokenizer,
                              model_max_tokens=model_max_tokens,
                              max_chunks=MAX_JD_CHUNKS,
                              source_label='JOB')
        for r, j in select_top_chunk_pairs(rc, jc, max_pairs):
            examples.append(InputExample(texts=[r, j],
                                         label=float(row['ground_truth'])))
        if (idx + 1) % 500 == 0:
            print(f'  Built examples for {idx + 1}/{total_rows} rows '
                  f'({len(examples)} chunk-pairs so far)')
    return examples

def aggregate_chunk_sims(sims, top_k=TOP_K_CHUNK_PAIRS):
    sims = np.asarray(sims, float)
    if sims.size == 0: return 0.0
    flat = sims.reshape(-1)
    k = min(top_k, len(flat))
    top_mean = float(np.partition(flat, -k)[-k:].mean())
    coverage = float((sims.max(axis=0).mean() + sims.max(axis=1).mean()) / 2)
    return float(np.clip(0.75 * top_mean + 0.25 * coverage, 0.0, 1.0))

def score_doc_pair(model, t1, t2, left='RESUME', right='JOB'):
    tok = model.tokenizer
    mlen = model.max_seq_length
    c1 = make_text_chunks(t1, tok, model_max_tokens=mlen,
                          max_chunks=MAX_RESUME_CHUNKS, source_label=left)
    c2 = make_text_chunks(t2, tok, model_max_tokens=mlen,
                          max_chunks=MAX_JD_CHUNKS, source_label=right)
    e1 = model.encode(c1, convert_to_tensor=True, normalize_embeddings=True,
                      batch_size=ENCODE_BATCH, show_progress_bar=False)
    e2 = model.encode(c2, convert_to_tensor=True, normalize_embeddings=True,
                      batch_size=ENCODE_BATCH, show_progress_bar=False)
    sims = util.cos_sim(e1, e2).detach().cpu().numpy()
    return aggregate_chunk_sims(sims)

class DocEvaluator(SentenceEvaluator):
    def __init__(self, frame, save_path, name='val'):
        self.frame = frame.reset_index(drop=True)
        self.save_path = save_path
        self.name = name
        self.best = -np.inf

    def __call__(self, model, output_path=None, epoch=-1, steps=-1):
        preds = [score_doc_pair(model, r['resume_raw'], r['jd_raw'])
                 for _, r in self.frame.iterrows()]
        yt = self.frame['ground_truth'].to_numpy(float)
        yp = np.asarray(preds, float)
        pe = float(pearsonr(yp, yt)[0]) if (len(yt) > 1 and np.std(yt) > 0 and np.std(yp) > 0) else 0.0
        sp = float(spearmanr(yp, yt).correlation) if len(yt) > 1 else 0.0
        pe = 0.0 if np.isnan(pe) else pe
        sp = 0.0 if np.isnan(sp) else sp
        mae = mean_absolute_error(yt, yp)
        print(f'[{self.name}] Pearson={pe:+.4f} Spearman={sp:+.4f} MAE={mae:.4f}')
        if pe > self.best:
            self.best = pe
            model.save(self.save_path)
            print(f'  Saved best checkpoint -> {self.save_path}')
        return pe

print('Chunking infrastructure ready')

## Train Model B

In [ ]:
ROBERTA_NAME = 'sentence-transformers/all-roberta-large-v1'
ROBERTA_PATH = os.path.join(MODEL_DIR, 'ezhire-roberta')
EPOCHS_B = 4
BATCH_B = 2 if DEVICE == 'cuda' else 2
LR_B = 2e-5
WARMUP_B = 100
WD_B = 0.01

print(f'Loading {ROBERTA_NAME}...')
roberta_model = SentenceTransformer(ROBERTA_NAME, device=DEVICE)
roberta_model.max_seq_length = 384

print('Building chunked training examples for roberta...')
roberta_train_ex = build_chunked_examples(df_tr, roberta_model.tokenizer, MAX_TRAIN_PAIRS)
print(f'Training examples: {len(roberta_train_ex)}')

roberta_loader = DataLoader(roberta_train_ex, shuffle=True, batch_size=BATCH_B)
roberta_loss = losses.CosineSimilarityLoss(roberta_model)

val_sample_b = df_vl.sample(n=min(DOC_EVAL_N, len(df_vl)), random_state=RANDOM_SEED)
roberta_eval = DocEvaluator(val_sample_b, save_path=ROBERTA_PATH, name='roberta_val')

total_steps_b = len(roberta_loader) * EPOCHS_B
warmup_b = min(WARMUP_B, max(1, total_steps_b // 10))
eval_steps_b = max(100, len(roberta_loader) // 2)

print(f'Training: {EPOCHS_B} epochs | {len(roberta_train_ex)} pairs | batch {BATCH_B} | warmup {warmup_b}')

roberta_model.fit(
    train_objectives=[(roberta_loader, roberta_loss)],
    evaluator=roberta_eval,
    epochs=EPOCHS_B,
    warmup_steps=warmup_b,
    optimizer_params={'lr': LR_B},
    weight_decay=WD_B,
    evaluation_steps=eval_steps_b,
    output_path=None,
    save_best_model=False,
    show_progress_bar=True,
    use_amp=(DEVICE == 'cuda')
)

if not os.path.exists(os.path.join(ROBERTA_PATH, 'modules.json')):
    roberta_model.save(ROBERTA_PATH)

roberta_model = SentenceTransformer(ROBERTA_PATH, device=DEVICE)
roberta_model.max_seq_length = 384
print('Model B (roberta) fine-tuning complete')

In [ ]:
print('Evaluating Model B (roberta) on full validation set...')
roberta_preds = []
for i, row in df_vl.iterrows():
    roberta_preds.append(score_doc_pair(roberta_model, row['resume_raw'], row['jd_raw']))
    if (i + 1) % 50 == 0: print(f'  {i + 1}/{len(df_vl)}')

df_vl['roberta_score'] = [round(s * 100, 2) for s in roberta_preds]
yt = df_vl['ground_truth'].values
yp = df_vl['roberta_score'].values
roberta_pearson = float(pearsonr(yp / 100, yt)[0])
roberta_spearman = float(spearmanr(yp / 100, yt).correlation)
roberta_mae = float(mean_absolute_error(yt, yp / 100))
roberta_rmse = float(np.sqrt(np.mean((denormalize_score(yt) - denormalize_score(yp / 100)) ** 2)))
roberta_r2 = float(r2_score(yt, yp / 100))

print('')
print('Model B (roberta) Results:')
print(f'  Pearson  : {roberta_pearson:+.4f}')
print(f'  Spearman : {roberta_spearman:+.4f}')
print(f'  MAE_norm : {roberta_mae:.4f}')
print(f'  RMSE_raw : {roberta_rmse:.2f}')
print(f'  R2_raw   : {roberta_r2:+.4f}')